# TwoTower (loopback): Last.fm 1K и MovieLens-100k

Алгоритм: [DanielRegaladoCardoso/loopback-twotower](https://huggingface.co/DanielRegaladoCardoso/loopback-twotower).

1. Воспроизводим результаты авторов на их демонстрационном датасете Last.fm 1K.
2. Обучаем тот же алгоритм на принципиально другом датасете, MovieLens-100k.
3. Сравниваем: что заявляли авторы и что получилось у нас.
4. Запускаем bash-скрипт «веса + датасет → вероятности следующего объекта».

Ноутбук только запускает bash-скрипты из репозитория, вся логика лежит в `twotower/` и `scripts/`.

**Перед запуском:** «Среда выполнения» → «Сменить среду выполнения» → **T4 GPU**. Весь ноутбук на T4 занимает около получаса; без видеокарты обучение на Last.fm растянется на часы.

## 0. Репозиторий и окружение

In [ ]:
%cd /content
!test -d rec_sys && git -C rec_sys pull --ff-only || git clone https://github.com/Sandrolek/rec_sys.git
%cd /content/rec_sys
# в Colab torch с CUDA уже установлен, отдельный venv не нужен
%env USE_SYSTEM_PYTHON=1
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo "Видеокарта не подключена: включите T4 в настройках среды выполнения"
!bash -c 'source scripts/env.sh && "$PY" -c "import torch; print(torch.__version__, torch.cuda.is_available())"'

## 1. Демонстрационный датасет: Last.fm 1K

### 1.1. Оценка опубликованных весов авторов

Скачиваются датасет (190 МБ) и чекпойнт авторов (430 МБ). Веса оцениваются сразу в нескольких вариантах протокола: авторы не опубликовали код оценки, и точное определение их Recall@K приходится восстанавливать.

In [ ]:
!bash scripts/reproduce_lastfm.sh eval

### 1.2. Обучение с нуля с гиперпараметрами авторов

3 эпохи, батч 4096, AdamW, lr 1e-3, weight decay 1e-5. Авторы сообщают loss 5.6 в конце обучения.

In [ ]:
!bash scripts/reproduce_lastfm.sh train

## 2. Второй датасет: MovieLens-100k

Два разбиения по времени: `global` (как у авторов, общая временная шкала) и `user` (внутри истории каждого пользователя). Для каждого — гиперпараметры авторов как есть, подбор размера батча по val с ранней остановкой и тот же запуск с поправкой logQ, которой у авторов нет. Занимает около 10 минут.

In [ ]:
!bash scripts/train_movielens.sh

## 3. Сводный отчёт: заявлено авторами и получено нами

In [ ]:
from IPython.display import Markdown

!python -m twotower.report > /dev/null
Markdown(open("results/REPORT.md").read())

## 4. Bash-скрипт: веса + датасет → вероятности следующего объекта

Скрипт сам готовит окружение, скачивает веса и данные, берёт несколько пользователей из test и считает вероятность каждого объекта каталога оказаться следующим.

In [ ]:
!bash scripts/predict.sh movielens 3

In [ ]:
# те же пользователи, но веса модели с поправкой logQ
!WEIGHTS=weights/twotower_ml100k_logq.pt bash scripts/predict.sh movielens 3

In [ ]:
# Last.fm: по умолчанию веса авторов; свои веса из пункта 1.2 — WEIGHTS=weights/twotower_lastfm.pt
!bash scripts/predict.sh lastfm 3

## 5. Забрать результаты

Архив с метриками, отчётом и весами для MovieLens (веса для Last.fm весят 430 МБ и в архив не входят). Содержимое архива нужно распаковать в корень репозитория и закоммитить.

In [ ]:
from google.colab import files

!rm -f results.zip && zip -qr results.zip results weights/twotower_ml100k.pt weights/twotower_ml100k_logq.pt
files.download("results.zip")